This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch torchvision --upgrade -q

## Classification and regression

### Classifying movie reviews: A binary classification example

#### The IMDb dataset

In [ ]:
# PyTorch: there's no keras.datasets. We download the same npz file Keras uses
# (numpy word-index arrays), so train_data/test_data have the identical shapes
# (object arrays of variable-length int lists) the rest of the chapter expects.
import numpy as np
from urllib.request import urlretrieve

def load_imdb(num_words=None, index_from=3, start_char=1, oov_char=2, skip_top=0):
    path = "./imdb.npz"
    urlretrieve("https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb.npz", path)
    with np.load(path, allow_pickle=True) as f:
        x_train, labels_train = f["x_train"], f["y_train"]
        x_test, labels_test = f["x_test"], f["y_test"]
    xs = np.concatenate([x_train, x_test])
    labels = np.concatenate([labels_train, labels_test])
    xs = [[start_char] + [w + index_from for w in x] for x in xs]
    if num_words is None:
        num_words = max(max(x) for x in xs)
    xs = [[w if (skip_top <= w < num_words) else oov_char for w in x] for x in xs]
    idx = len(x_train)
    x_train = np.array(xs[:idx], dtype="object")
    x_test = np.array(xs[idx:], dtype="object")
    return (x_train, labels_train), (x_test, labels_test)

(train_data, train_labels), (test_data, test_labels) = load_imdb(num_words=10000)

In [0]:
train_data[0]

In [0]:
train_labels[0]

In [0]:
max([max(sequence) for sequence in train_data])

In [ ]:
# PyTorch: fetch the same word-index JSON Keras serves (maps words -> int rank).
import json as _json
urlretrieve("https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb_word_index.json", "./imdb_word_index.json")
word_index = _json.load(open("./imdb_word_index.json"))
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])
decoded_review = " ".join(
    [reverse_word_index.get(i - 3, "?") for i in train_data[0]]
)

In [0]:
decoded_review[:100]

#### Preparing the data

In [0]:
import numpy as np

def multi_hot_encode(sequences, num_classes):
    results = np.zeros((len(sequences), num_classes))
    for i, sequence in enumerate(sequences):
        results[i][sequence] = 1.0
    return results

x_train = multi_hot_encode(train_data, num_classes=10000)
x_test = multi_hot_encode(test_data, num_classes=10000)

In [0]:
x_train[0]

In [0]:
y_train = train_labels.astype("float32")
y_test = test_labels.astype("float32")

#### Building your model

In [ ]:
import torch
from torch import nn

# PyTorch: nn.Sequential needs explicit in/out sizes (no lazy build), and we drop
# the final sigmoid because nn.BCEWithLogitsLoss expects raw logits.
model = nn.Sequential(
    nn.Linear(10000, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

In [ ]:
# PyTorch: no compile() step -- create the optimizer and loss directly.
# nn.BCEWithLogitsLoss = sigmoid + binary crossentropy on logits.
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()

#### Validating your approach

In [0]:
x_val = x_train[:10000]
partial_x_train = x_train[10000:]
y_val = y_train[:10000]
partial_y_train = y_train[10000:]

In [ ]:
# PyTorch: model.fit() becomes an explicit training loop. We batch the data
# ourselves and, because validation_data was given, run a no-grad validation pass
# each epoch, recording loss/accuracy into a history dict like Keras returned.
def train_model(model, optimizer, loss_fn, x, y, epochs, batch_size,
                validation_data=None, verbose=1):
    x = torch.as_tensor(np.asarray(x), dtype=torch.float32)
    y = torch.as_tensor(np.asarray(y), dtype=torch.float32).reshape(-1, 1)
    if validation_data is not None:
        xv, yv = validation_data
        xv = torch.as_tensor(np.asarray(xv), dtype=torch.float32)
        yv = torch.as_tensor(np.asarray(yv), dtype=torch.float32).reshape(-1, 1)
    history = {"loss": [], "accuracy": []}
    if validation_data is not None:
        history["val_loss"], history["val_accuracy"] = [], []
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(x))
        for i in range(0, len(x), batch_size):
            idx = perm[i : i + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(model(x[idx]), y[idx])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            logits = model(x)
            history["loss"].append(loss_fn(logits, y).item())
            history["accuracy"].append(((logits > 0).float() == y).float().mean().item())
            if validation_data is not None:
                vlogits = model(xv)
                history["val_loss"].append(loss_fn(vlogits, yv).item())
                history["val_accuracy"].append(((vlogits > 0).float() == yv).float().mean().item())
        if verbose:
            print(f"Epoch {epoch + 1}/{epochs} - loss {history['loss'][-1]:.4f}")
    return history

history = train_model(
    model, optimizer, loss_fn,
    partial_x_train, partial_y_train,
    epochs=20, batch_size=512,
    validation_data=(x_val, y_val),
)

In [ ]:
# PyTorch: there's no validation_split argument; we carve off the last 20% of the
# training data as the validation set ourselves, then reuse train_model.
val_count = int(len(x_train) * 0.2)
history = train_model(
    model, optimizer, loss_fn,
    x_train[:-val_count], y_train[:-val_count],
    epochs=20, batch_size=512,
    validation_data=(x_train[-val_count:], y_train[-val_count:]),
)

In [ ]:
# PyTorch: train_model already returns a plain history dict (no .history attr).
history_dict = history
history_dict.keys()

In [ ]:
import matplotlib.pyplot as plt

history_dict = history
loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]
epochs = range(1, len(loss_values) + 1)
plt.plot(epochs, loss_values, "r--", label="Training loss")
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.title("[IMDB] Training and validation loss")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Loss")
plt.legend()
plt.show()

In [0]:
plt.clf()
acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]
plt.plot(epochs, acc, "r--", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("[IMDB] Training and validation accuracy")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
model = nn.Sequential(
    nn.Linear(10000, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()
train_model(model, optimizer, loss_fn, x_train, y_train, epochs=4, batch_size=512)

# PyTorch: model.evaluate() -> manual no-grad pass returning [loss, accuracy].
def evaluate_binary(model, x, y):
    x = torch.as_tensor(np.asarray(x), dtype=torch.float32)
    y = torch.as_tensor(np.asarray(y), dtype=torch.float32).reshape(-1, 1)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = loss_fn(logits, y).item()
        acc = ((logits > 0).float() == y).float().mean().item()
    return [loss, acc]

results = evaluate_binary(model, x_test, y_test)

In [0]:
results

#### Using a trained model to generate predictions on new data

In [ ]:
# PyTorch: model.predict() -> no-grad forward; apply sigmoid to get probabilities
# (Keras applied sigmoid inside the model).
model.eval()
with torch.no_grad():
    torch.sigmoid(model(torch.as_tensor(np.asarray(x_test), dtype=torch.float32))).numpy()

#### Further experiments

#### Wrapping up

### Classifying newswires: A multiclass classification example

#### The Reuters dataset

In [ ]:
# PyTorch: download the same npz Keras serves for Reuters (numpy word-index
# arrays), keeping identical shapes/semantics (num_words top-frequency filter).
def load_reuters(num_words=None, index_from=3, start_char=1, oov_char=2, skip_top=0,
                 test_split=0.2, seed=113):
    path = "./reuters.npz"
    urlretrieve("https://storage.googleapis.com/tensorflow/tf-keras-datasets/reuters.npz", path)
    with np.load(path, allow_pickle=True) as f:
        xs, labels = f["x"], f["y"]
    rng = np.random.RandomState(seed)
    indices = np.arange(len(xs))
    rng.shuffle(indices)
    xs, labels = xs[indices], labels[indices]
    xs = [[start_char] + [w + index_from for w in x] for x in xs]
    if num_words is None:
        num_words = max(max(x) for x in xs)
    xs = [[w if (skip_top <= w < num_words) else oov_char for w in x] for x in xs]
    idx = int(len(xs) * (1 - test_split))
    x_train = np.array(xs[:idx], dtype="object")
    x_test = np.array(xs[idx:], dtype="object")
    return (x_train, labels[:idx]), (x_test, labels[idx:])

(train_data, train_labels), (test_data, test_labels) = load_reuters(num_words=10000)

In [0]:
len(train_data)

In [0]:
len(test_data)

In [0]:
train_data[10]

In [ ]:
# PyTorch: fetch the same Reuters word-index JSON Keras serves.
urlretrieve("https://storage.googleapis.com/tensorflow/tf-keras-datasets/reuters_word_index.json", "./reuters_word_index.json")
word_index = _json.load(open("./reuters_word_index.json"))
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])
decoded_newswire = " ".join(
    [reverse_word_index.get(i - 3, "?") for i in train_data[10]]
)

In [0]:
train_labels[10]

#### Preparing the data

In [0]:
x_train = multi_hot_encode(train_data, num_classes=10000)
x_test = multi_hot_encode(test_data, num_classes=10000)

In [0]:
def one_hot_encode(labels, num_classes=46):
    results = np.zeros((len(labels), num_classes))
    for i, label in enumerate(labels):
        results[i, label] = 1.0
    return results

y_train = one_hot_encode(train_labels)
y_test = one_hot_encode(test_labels)

In [ ]:
# PyTorch: keras.utils.to_categorical -> one-hot via numpy (same as one_hot_encode).
y_train = one_hot_encode(train_labels)
y_test = one_hot_encode(test_labels)

#### Building your model

In [ ]:
# PyTorch: drop the final softmax; nn.CrossEntropyLoss expects raw logits.
model = nn.Sequential(
    nn.Linear(10000, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 46),
)

In [ ]:
# PyTorch: no compile(); use Adam + CrossEntropyLoss (= softmax + categorical
# crossentropy on logits). We compute accuracy and top-3 accuracy manually in the
# training loop below instead of passing metric objects.
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

#### Validating your approach

In [0]:
x_val = x_train[:1000]
partial_x_train = x_train[1000:]
y_val = y_train[:1000]
partial_y_train = y_train[1000:]

In [ ]:
# PyTorch: explicit multiclass training loop. Targets here are one-hot, so we take
# argmax to get class indices for CrossEntropyLoss and for accuracy/top-3 metrics.
def train_multiclass(model, optimizer, loss_fn, x, y, epochs, batch_size,
                     validation_data=None, verbose=1):
    x = torch.as_tensor(np.asarray(x), dtype=torch.float32)
    y = torch.as_tensor(np.asarray(y))
    y_idx = y.argmax(dim=1).long() if y.ndim > 1 else y.long()
    if validation_data is not None:
        xv, yv = validation_data
        xv = torch.as_tensor(np.asarray(xv), dtype=torch.float32)
        yv = torch.as_tensor(np.asarray(yv))
        yv_idx = yv.argmax(dim=1).long() if yv.ndim > 1 else yv.long()

    def metrics(logits, targets):
        preds = logits.argmax(dim=1)
        acc = (preds == targets).float().mean().item()
        top3 = logits.topk(3, dim=1).indices.eq(targets.unsqueeze(1)).any(dim=1).float().mean().item()
        return acc, top3

    history = {"loss": [], "accuracy": [], "top_3_accuracy": []}
    if validation_data is not None:
        history.update({"val_loss": [], "val_accuracy": [], "val_top_3_accuracy": []})
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(x))
        for i in range(0, len(x), batch_size):
            idx = perm[i : i + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(model(x[idx]), y_idx[idx])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            logits = model(x)
            history["loss"].append(loss_fn(logits, y_idx).item())
            acc, top3 = metrics(logits, y_idx)
            history["accuracy"].append(acc)
            history["top_3_accuracy"].append(top3)
            if validation_data is not None:
                vlogits = model(xv)
                history["val_loss"].append(loss_fn(vlogits, yv_idx).item())
                vacc, vtop3 = metrics(vlogits, yv_idx)
                history["val_accuracy"].append(vacc)
                history["val_top_3_accuracy"].append(vtop3)
        if verbose:
            print(f"Epoch {epoch + 1}/{epochs} - loss {history['loss'][-1]:.4f}")
    return history

history = train_multiclass(
    model, optimizer, loss_fn,
    partial_x_train, partial_y_train,
    epochs=20, batch_size=512,
    validation_data=(x_val, y_val),
)

In [ ]:
loss = history["loss"]
val_loss = history["val_loss"]
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, "r--", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.clf()
acc = history["accuracy"]
val_acc = history["val_accuracy"]
plt.plot(epochs, acc, "r--", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.clf()
acc = history["top_3_accuracy"]
val_acc = history["val_top_3_accuracy"]
plt.plot(epochs, acc, "r--", label="Training top-3 accuracy")
plt.plot(epochs, val_acc, "b", label="Validation top-3 accuracy")
plt.title("Training and validation top-3 accuracy")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Top-3 accuracy")
plt.legend()
plt.show()

In [ ]:
model = nn.Sequential(
    nn.Linear(10000, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 46),
)
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()
train_multiclass(model, optimizer, loss_fn, x_train, y_train, epochs=9, batch_size=512)

# PyTorch: model.evaluate() -> manual no-grad pass returning [loss, accuracy].
def evaluate_multiclass(model, x, y):
    x = torch.as_tensor(np.asarray(x), dtype=torch.float32)
    y = torch.as_tensor(np.asarray(y))
    y_idx = y.argmax(dim=1).long() if y.ndim > 1 else y.long()
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = loss_fn(logits, y_idx).item()
        acc = (logits.argmax(dim=1) == y_idx).float().mean().item()
    return [loss, acc]

results = evaluate_multiclass(model, x_test, y_test)

In [0]:
results

In [0]:
import copy
test_labels_copy = copy.copy(test_labels)
np.random.shuffle(test_labels_copy)
hits_array = np.array(test_labels == test_labels_copy)
hits_array.mean()

#### Generating predictions on new data

In [ ]:
# PyTorch: model.predict() -> no-grad forward + softmax to recover probabilities.
model.eval()
with torch.no_grad():
    predictions = torch.softmax(
        model(torch.as_tensor(np.asarray(x_test), dtype=torch.float32)), dim=1
    ).numpy()

In [0]:
predictions[0].shape

In [0]:
np.sum(predictions[0])

In [0]:
np.argmax(predictions[0])

#### A different way to handle the labels and the loss

In [0]:
y_train = train_labels
y_test = test_labels

In [ ]:
# PyTorch: with integer labels (y_train = train_labels), nn.CrossEntropyLoss is
# already the sparse-categorical equivalent -- it takes int class indices directly,
# so no change to the loss is needed (train_multiclass handles 1-D int targets).
loss_fn = nn.CrossEntropyLoss()

#### The importance of having sufficiently large intermediate layers

In [ ]:
# PyTorch: 4-unit bottleneck layer; same drop-the-softmax convention.
model = nn.Sequential(
    nn.Linear(10000, 64),
    nn.ReLU(),
    nn.Linear(64, 4),
    nn.ReLU(),
    nn.Linear(4, 46),
)
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()
train_multiclass(
    model, optimizer, loss_fn,
    partial_x_train, partial_y_train,
    epochs=20, batch_size=128,
    validation_data=(x_val, y_val),
)

#### Further experiments

#### Wrapping up

### Predicting house prices: A regression example

#### The California Housing Price dataset

In [ ]:
# PyTorch: keras.datasets.california_housing serves a prebuilt npz. We download the
# same "small" split npz Keras uses so train_data/test_data shapes match exactly.
def load_california_housing(version="small", test_split=0.2, seed=113):
    fname = "california_housing.npz" if version == "large" else "california_housing_small.npz"
    path = "./" + fname
    urlretrieve("https://storage.googleapis.com/tensorflow/tf-keras-datasets/" + fname, path)
    with np.load(path, allow_pickle=True) as f:
        x, y = f["x"], f["y"]
    rng = np.random.RandomState(seed)
    indices = np.arange(len(x))
    rng.shuffle(indices)
    x, y = x[indices], y[indices]
    idx = int(len(x) * (1 - test_split))
    return (x[:idx], y[:idx]), (x[idx:], y[idx:])

(train_data, train_targets), (test_data, test_targets) = load_california_housing(version="small")

In [0]:
train_data.shape

In [0]:
test_data.shape

In [0]:
train_targets

#### Preparing the data

In [0]:
mean = train_data.mean(axis=0)
std = train_data.std(axis=0)
x_train = (train_data - mean) / std
x_test = (test_data - mean) / std

In [0]:
y_train = train_targets / 100000
y_test = test_targets / 100000

#### Building your model

In [ ]:
# PyTorch: get_model returns the model together with its optimizer and loss, since
# there's no compile() to attach them. loss=MSELoss, and we track MAE separately.
def get_model():
    model = nn.Sequential(
        nn.Linear(train_data.shape[1], 64),
        nn.ReLU(),
        nn.Linear(64, 64),
        nn.ReLU(),
        nn.Linear(64, 1),
    )
    optimizer = torch.optim.Adam(model.parameters())
    loss_fn = nn.MSELoss()
    return model, optimizer, loss_fn

#### Validating your approach using K-fold validation

In [ ]:
# PyTorch: regression training loop (MSELoss) + evaluate returning [mse, mae].
def train_regression(model, optimizer, loss_fn, x, y, epochs, batch_size,
                     validation_data=None, verbose=0):
    x = torch.as_tensor(np.asarray(x), dtype=torch.float32)
    y = torch.as_tensor(np.asarray(y), dtype=torch.float32).reshape(-1, 1)
    if validation_data is not None:
        xv, yv = validation_data
        xv = torch.as_tensor(np.asarray(xv), dtype=torch.float32)
        yv = torch.as_tensor(np.asarray(yv), dtype=torch.float32).reshape(-1, 1)
    history = {"loss": [], "mean_absolute_error": []}
    if validation_data is not None:
        history.update({"val_loss": [], "val_mean_absolute_error": []})
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(x))
        for i in range(0, len(x), batch_size):
            idx = perm[i : i + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(model(x[idx]), y[idx])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            preds = model(x)
            history["loss"].append(loss_fn(preds, y).item())
            history["mean_absolute_error"].append((preds - y).abs().mean().item())
            if validation_data is not None:
                vpreds = model(xv)
                history["val_loss"].append(loss_fn(vpreds, yv).item())
                history["val_mean_absolute_error"].append((vpreds - yv).abs().mean().item())
        if verbose:
            print(f"Epoch {epoch + 1}/{epochs} - loss {history['loss'][-1]:.4f}")
    return history

def evaluate_regression(model, loss_fn, x, y):
    x = torch.as_tensor(np.asarray(x), dtype=torch.float32)
    y = torch.as_tensor(np.asarray(y), dtype=torch.float32).reshape(-1, 1)
    model.eval()
    with torch.no_grad():
        preds = model(x)
        mse = loss_fn(preds, y).item()
        mae = (preds - y).abs().mean().item()
    return [mse, mae]

k = 4
num_val_samples = len(x_train) // k
num_epochs = 50
all_scores = []
for i in range(k):
    print(f"Processing fold #{i + 1}")
    fold_x_val = x_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_y_val = y_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_x_train = np.concatenate(
        [x_train[: i * num_val_samples], x_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    fold_y_train = np.concatenate(
        [y_train[: i * num_val_samples], y_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    model, optimizer, loss_fn = get_model()
    train_regression(
        model, optimizer, loss_fn,
        fold_x_train, fold_y_train,
        epochs=num_epochs, batch_size=16, verbose=0,
    )
    scores = evaluate_regression(model, loss_fn, fold_x_val, fold_y_val)
    val_loss, val_mae = scores
    all_scores.append(val_mae)

In [0]:
[round(value, 3) for value in all_scores]

In [0]:
round(np.mean(all_scores), 3)

In [ ]:
k = 4
num_val_samples = len(x_train) // k
num_epochs = 200
all_mae_histories = []
for i in range(k):
    print(f"Processing fold #{i + 1}")
    fold_x_val = x_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_y_val = y_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_x_train = np.concatenate(
        [x_train[: i * num_val_samples], x_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    fold_y_train = np.concatenate(
        [y_train[: i * num_val_samples], y_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    model, optimizer, loss_fn = get_model()
    history = train_regression(
        model, optimizer, loss_fn,
        fold_x_train, fold_y_train,
        validation_data=(fold_x_val, fold_y_val),
        epochs=num_epochs, batch_size=16, verbose=0,
    )
    mae_history = history["val_mean_absolute_error"]
    all_mae_histories.append(mae_history)

In [0]:
average_mae_history = [
    np.mean([x[i] for x in all_mae_histories]) for i in range(num_epochs)
]

In [0]:
epochs = range(1, len(average_mae_history) + 1)
plt.plot(epochs, average_mae_history)
plt.xlabel("Epochs")
plt.ylabel("Validation MAE")
plt.show()

In [0]:
truncated_mae_history = average_mae_history[10:]
epochs = range(10, len(truncated_mae_history) + 10)
plt.plot(epochs, truncated_mae_history)
plt.xlabel("Epochs")
plt.ylabel("Validation MAE")
plt.show()

In [ ]:
model, optimizer, loss_fn = get_model()
train_regression(model, optimizer, loss_fn, x_train, y_train, epochs=130, batch_size=16, verbose=0)
test_mean_squared_error, test_mean_absolute_error = evaluate_regression(
    model, loss_fn, x_test, y_test
)

In [0]:
round(test_mean_absolute_error, 3)

#### Generating predictions on new data

In [ ]:
# PyTorch: regression predict -> no-grad forward.
model.eval()
with torch.no_grad():
    predictions = model(torch.as_tensor(np.asarray(x_test), dtype=torch.float32)).numpy()
predictions[0]

#### Wrapping up